# Creating the `partido_regional` variable

**Goal:** For each party competing in the 2018 district-level municipal election, determine whether it also competed in the regional (governor) election in the same department.

- `partido_regional = 1` if the party also competes in the regional election
- `partido_regional = 0` if the party only competes at the district level

**Data sources:**
- District-level results: `resultados_onpe_erm2018_por_distrito_distrital.csv`
- Regional results: `ERM2018_Regional_total.csv` (downloaded from datos abiertos)

In [ ]:
import pandas as pd
import unicodedata

## 1. Load data

In [ ]:
dis = pd.read_csv('resultados_onpe_erm2018_por_distrito_distrital.csv', encoding='utf-8-sig')
reg = pd.read_csv('ERM2018_Regional_total.csv', sep=';', encoding='latin-1')

print(f'District-level: {len(dis):,} rows, {dis.columns.tolist()}')
print(f'Regional:       {len(reg):,} rows, {reg.columns.tolist()}')

In [ ]:
# Remove summary rows
dis = dis[dis['organizacion_politica'] != 'TOTAL DE VOTOS VALIDOS'].copy()
print(f'District-level after filtering: {len(dis):,} rows')

## 2. Normalize names

The regional file uses uppercase without accents (`AMAZONAS`), while the district file uses title case with accents (`Amazonas`, `Apurímac`). We normalize by removing accents and converting to uppercase for matching.

In [ ]:
def normalize_name(s):
    """Remove accents and convert to uppercase for matching."""
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.strip().upper()

In [ ]:
dis['depto_norm'] = dis['region'].apply(normalize_name)
reg['depto_norm'] = reg['DEPARTAMENTO'].apply(normalize_name)

dis['partido_norm'] = dis['organizacion_politica'].apply(normalize_name)
reg['partido_norm'] = reg['AGRUPACION_POLITICA'].apply(normalize_name)

# Verify all 25 departments match
assert set(dis['depto_norm'].unique()) == set(reg['depto_norm'].unique()), \
    'Department mismatch!'
print(f'All 25 departments match between district and regional data.')

## 3. Create `partido_regional`

Build a set of `(department, party)` pairs from the regional election, then flag each district-level row.

In [ ]:
regional_set = set(zip(reg['depto_norm'], reg['partido_norm']))
print(f'Unique (department, party) pairs in regional election: {len(regional_set)}')

dis['partido_regional'] = [
    1 if (d, p) in regional_set else 0
    for d, p in zip(dis['depto_norm'], dis['partido_norm'])
]

print(f'\npartido_regional distribution:')
print(dis['partido_regional'].value_counts())
print(f'\nProportion competing in regional: {dis["partido_regional"].mean():.3f}')

## 4. Summary by department

In [ ]:
summary = []
for depto in sorted(dis['depto_norm'].unique()):
    sub = dis[dis['depto_norm'] == depto]
    n_total = sub['partido_norm'].nunique()
    n_reg = sub[sub['partido_regional'] == 1]['partido_norm'].nunique()
    n_only = sub[sub['partido_regional'] == 0]['partido_norm'].nunique()
    summary.append({'department': depto, 'total_parties': n_total,
                    'in_regional': n_reg, 'district_only': n_only})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

## 5. Save output

In [ ]:
# Drop auxiliary columns
out = dis.drop(columns=['depto_norm', 'partido_norm'])

out_path = 'resultados_onpe_erm2018_distrital_con_partido_regional.csv'
out.to_csv(out_path, index=False, encoding='latin-1')
print(f'Saved: {out_path}')
print(f'Shape: {out.shape}')
print(f'Columns: {out.columns.tolist()}')